In [2]:
 !pip install yfinance statsmodels -q

In [6]:
"""
PROJECT 5: Time Series Forecasting (Stock Price Example)
-------------------------------------------------------------
Goal: Forecast future stock closing prices.
Dataset: Downloaded live via yfinance (change TICKER to any stock symbol).

Run:
    pip install -r requirements.txt
    python 05_time_series_forecasting.py
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

TICKER = "AAPL"
START_DATE = "2018-01-01"
END_DATE = "2024-12-31"
N_LAGS = 5


def load_data():
    print(f"Downloading {TICKER} data from {START_DATE} to {END_DATE}...")
    df = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)
    # Recent yfinance versions can return columns as a MultiIndex
    # (e.g. ('Close', 'AAPL')) even for a single ticker. Flatten to
    # plain column names so downstream code always gets a clean Series.
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[["Close"]].dropna()
    # yfinance returns a plain DatetimeIndex with no explicit frequency, which
    # newer statsmodels versions require for forecasting. Set it to business-day
    # frequency and forward-fill any gaps (holidays) this creates.
    df = df.asfreq("B")
    df["Close"] = df["Close"].ffill()
    print("Shape:", df.shape)
    return df


def plot_series(df):
    plt.figure(figsize=(10, 5))
    plt.plot(df.index, df["Close"])
    plt.title(f"{TICKER} Closing Price")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.tight_layout()
    plt.savefig("price_series.png")
    plt.close()


def check_stationarity(series):
    result = adfuller(series.dropna())
    print(f"\nADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    if result[1] < 0.05:
        print("Series is likely stationary.")
    else:
        print("Series is likely non-stationary (ARIMA will handle differencing with 'd').")


def naive_baseline(df):
    df = df.copy()
    df["naive_pred"] = df["Close"].shift(1)
    valid = df.dropna()
    mae = mean_absolute_error(valid["Close"], valid["naive_pred"])
    print(f"\nNaive baseline (yesterday's price) MAE: {mae:.4f}")
    return mae


def arima_forecast(train, test):
    print("\nFitting SARIMAX (ARIMA) model...")
    model = SARIMAX(train["Close"], order=(5, 1, 0))
    fitted = model.fit(disp=False)
    forecast = fitted.forecast(steps=len(test))
    mae = mean_absolute_error(test["Close"], forecast)
    rmse = np.sqrt(mean_squared_error(test["Close"], forecast))
    print(f"ARIMA MAE: {mae:.4f}  RMSE: {rmse:.4f}")
    return forecast, mae, rmse


def make_lag_features(df, n_lags=N_LAGS):
    df = df.copy()
    for lag in range(1, n_lags + 1):
        df[f"lag_{lag}"] = df["Close"].shift(lag)
    df = df.dropna()
    return df


def ml_forecast(df_feat, train_size):
    feature_cols = [c for c in df_feat.columns if c.startswith("lag_")]
    X, y = df_feat[feature_cols], df_feat["Close"]
    X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
    y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

    model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"Random Forest (lag features) MAE: {mae:.4f}  RMSE: {rmse:.4f}")
    return preds, y_test, mae, rmse


def walk_forward_validation(df_feat, n_splits=5):
    """Rolling-origin validation instead of a single train/test split."""
    feature_cols = [c for c in df_feat.columns if c.startswith("lag_")]
    X, y = df_feat[feature_cols].values, df_feat["Close"].values
    fold_size = len(X) // (n_splits + 1)

    maes = []
    for i in range(1, n_splits + 1):
        train_end = fold_size * i
        test_end = fold_size * (i + 1)
        X_train, y_train = X[:train_end], y[:train_end]
        X_test, y_test = X[train_end:test_end], y[train_end:test_end]
        if len(X_test) == 0:
            continue
        model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        mae = mean_absolute_error(y_test, preds)
        maes.append(mae)
        print(f"  Fold {i}: MAE = {mae:.4f}")

    print(f"Walk-forward average MAE: {np.mean(maes):.4f}")
    return maes


def main():
    df = load_data()
    plot_series(df)
    check_stationarity(df["Close"])
    naive_baseline(df)

    split_idx = int(len(df) * 0.8)
    train, test = df.iloc[:split_idx], df.iloc[split_idx:]

    arima_preds, arima_mae, arima_rmse = arima_forecast(train, test)

    df_feat = make_lag_features(df)
    ml_train_size = int(len(df_feat) * 0.8)
    ml_preds, y_test, ml_mae, ml_rmse = ml_forecast(df_feat, ml_train_size)

    plt.figure(figsize=(10, 5))
    plt.plot(test.index, test["Close"], label="Actual")
    plt.plot(test.index, arima_preds, label="ARIMA Forecast")
    plt.legend()
    plt.title(f"{TICKER} - ARIMA Forecast vs Actual")
    plt.tight_layout()
    plt.savefig("arima_forecast.png")
    plt.close()

    print("\nRunning walk-forward validation for the Random Forest model...")
    walk_forward_validation(df_feat)

    results = pd.DataFrame([
        {"model": "Naive", "MAE": naive_baseline(df)},
        {"model": "ARIMA", "MAE": arima_mae, "RMSE": arima_rmse},
        {"model": "Random Forest (lags)", "MAE": ml_mae, "RMSE": ml_rmse},
    ])
    print("\n--- Model comparison ---")
    print(results)
    results.to_csv("forecast_model_comparison.csv", index=False)


if __name__ == "__main__":
    main()

[*********************100%***********************]  1 of 1 completed

Shape: (1825, 1)


/tmp/ipykernel_1640/241146476.py:57: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  result = adfuller(series.dropna())



ADF Statistic: 0.1665
p-value: 0.9703
Series is likely non-stationary (ARIMA will handle differencing with 'd').

Naive baseline (yesterday's price) MAE: 1.5113

Fitting SARIMAX (ARIMA) model...
ARIMA MAE: 24.5461  RMSE: 33.0211
Random Forest (lag features) MAE: 16.1172  RMSE: 24.6709

Running walk-forward validation for the Random Forest model...
  Fold 1: MAE = 7.4181
  Fold 2: MAE = 36.4562
  Fold 3: MAE = 18.7343
  Fold 4: MAE = 3.9573
  Fold 5: MAE = 17.7042
Walk-forward average MAE: 16.8540

Naive baseline (yesterday's price) MAE: 1.5113

--- Model comparison ---
                  model        MAE       RMSE
0                 Naive   1.511307        NaN
1                 ARIMA  24.546110  33.021055
2  Random Forest (lags)  16.117152  24.670915
